# Figure S6: PCA selection and second-stage clustering diagnostics

This notebook reproduces the second-stage clustering diagnostics from the
matched PCS-state features. The four panels show: (a) PCA component selection,
(b) elbow-based selection of the number of K-means clusters, (c) the two-pattern
projection in the first two principal components, and (d) global SHAP feature
importance from a Random Forest surrogate trained to reproduce the K-means
labels.

The SHAP panel describes statistical separation learned by the surrogate model;
it does not establish a causal role for any feature. Plotting code is written
directly in one cell, with all typography and layout parameters at the top for
easy editing. No plotting functions are defined.

## Setup and analysis parameters

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from kneed import KneeLocator
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

# Locate the notebook directory whether Jupyter starts here or at project root.
paper_candidates = [
    Path.cwd(),
    Path.cwd() / "uilt_oneday_all" / "uilt_paper_figure",
    Path.cwd().parent,
]
PAPER_DIR = next(
    (
        candidate.resolve()
        for candidate in paper_candidates
        if (candidate / "FigureS6_second_clustering.ipynb").exists()
    ),
    Path(
        r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure"
    ),
)
PROJECT_ROOT = PAPER_DIR.parents[1]
SECONDARY_DIR = PROJECT_ROOT / "uilt_oneday_all" / "03visual" / "二次聚类"
INPUT_FILE = (
    SECONDARY_DIR
    / "matched_precursor_features"
    / "matched_precursor_features_with_distance.xlsx"
)
REFERENCE_RESULT_CSV = (
    SECONDARY_DIR
    / "kmeans_clustering_from_matched_precursor_features"
    / "kmeans_clustering_results.csv"
)
OUT_DIR = PAPER_DIR / "FigureS6_second_clustering_exports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Editable analysis parameters ----------
RANDOM_STATE = 42
K_VALUES = range(1, 7)
PCA_VARIANCE_THRESHOLD = 0.95
KMEANS_N_INIT = 100
REMOVE_HIGH_CORRELATION = True
CORRELATION_THRESHOLD = 0.95
MAX_MISSING_RATE = 0.50
RF_N_ESTIMATORS = 500
SHAP_TOP_N = 12
# --------------------------------------------------

print(f"Input features: {INPUT_FILE}")
print(f"Reference labels: {REFERENCE_RESULT_CSV}")
print(f"Figure output: {OUT_DIR}")

## Recompute PCA, K-means and SHAP inputs

In [ ]:
# Read the matched windows and reproduce the event-level PCS-state centers.
matched_df = pd.read_excel(INPUT_FILE)
matched_df["event"] = matched_df["event"].astype(str)
matched_df["timestamp"] = matched_df["timestamp"].astype(str)
if "selected_cluster_id" in matched_df.columns:
    matched_df = matched_df.rename(columns={"selected_cluster_id": "cluster_id"})
matched_df["cluster_id"] = matched_df["cluster_id"].astype(int)
matched_df["_event_number"] = pd.to_numeric(
    matched_df["event"].str.extract(r"(\d+)$")[0],
    errors="coerce",
).fillna(10**9)
matched_df = matched_df.sort_values(
    ["_event_number", "cluster_id", "Distance_to_Center"]
).drop(columns="_event_number").reset_index(drop=True)

metadata_for_center = {
    "event", "timestamp", "cluster_id", "raw_precursor_label",
    "Distance_to_Center", "x_file", "duration",
}
center_feature_columns = [
    column
    for column in matched_df.columns
    if column not in metadata_for_center
    and not str(column).startswith("PC")
    and pd.api.types.is_numeric_dtype(matched_df[column])
]
center_df = (
    matched_df.groupby(["event", "cluster_id"], as_index=False)
    .agg(
        sample_count=("timestamp", "count"),
        mean_distance_to_center=("Distance_to_Center", "mean"),
        min_distance_to_center=("Distance_to_Center", "min"),
        max_distance_to_center=("Distance_to_Center", "max"),
        **{column: (column, "mean") for column in center_feature_columns},
    )
)
center_df["_event_number"] = pd.to_numeric(
    center_df["event"].str.extract(r"(\d+)$")[0],
    errors="coerce",
).fillna(10**9)
center_df = center_df.sort_values("_event_number").drop(columns="_event_number").reset_index(drop=True)

# Apply the same feature-screening sequence as 08_kmeans_precursor2.py.
center_metadata = {
    "event", "cluster_id", "sample_count", "mean_distance_to_center",
    "min_distance_to_center", "max_distance_to_center",
}
raw_feature_columns = [
    column
    for column in center_df.columns
    if column not in center_metadata
    and not str(column).startswith("PC")
    and pd.api.types.is_numeric_dtype(center_df[column])
]
X_raw = center_df[raw_feature_columns].replace([np.inf, -np.inf], np.nan)
missing_rates = X_raw.isna().mean()
kept_after_missing = missing_rates[missing_rates <= MAX_MISSING_RATE].index.tolist()
X_imputed = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(X_raw[kept_after_missing]),
    columns=kept_after_missing,
    index=center_df.index,
)
nonconstant_columns = X_imputed.columns[X_imputed.std(axis=0) > 1e-12].tolist()
X_filtered = X_imputed[nonconstant_columns].copy()

dropped_correlated_columns = []
if REMOVE_HIGH_CORRELATION:
    absolute_correlation = X_filtered.corr().abs()
    upper_triangle = absolute_correlation.where(
        np.triu(np.ones(absolute_correlation.shape), k=1).astype(bool)
    )
    dropped_correlated_columns = [
        column
        for column in upper_triangle.columns
        if (upper_triangle[column] > CORRELATION_THRESHOLD).any()
    ]
    X_filtered = X_filtered.drop(columns=dropped_correlated_columns)

X_scaled = pd.DataFrame(
    StandardScaler().fit_transform(X_filtered),
    columns=X_filtered.columns,
    index=X_filtered.index,
)

# PCA component selection based on cumulative explained variance.
pca_full = PCA(random_state=RANDOM_STATE).fit(X_scaled)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
n_components = int(np.searchsorted(cumulative_variance, PCA_VARIANCE_THRESHOLD) + 1)
n_components = max(2, min(n_components, X_scaled.shape[0] - 1, X_scaled.shape[1]))
pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

# Evaluate K=1...6, then select the elbow exactly as in the clustering script.
score_rows = []
valid_k_values = [k for k in K_VALUES if 1 <= k <= len(center_df) - 1]
for k in valid_k_values:
    k_model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=KMEANS_N_INIT,
        init="k-means++",
    ).fit(X_pca)
    labels = k_model.labels_
    row = {
        "k": k,
        "inertia": float(k_model.inertia_),
        "min_cluster_size": int(pd.Series(labels).value_counts().min()),
    }
    if k >= 2 and len(np.unique(labels)) >= 2:
        row["silhouette"] = float(silhouette_score(X_pca, labels))
        row["calinski_harabasz"] = float(calinski_harabasz_score(X_pca, labels))
        row["davies_bouldin"] = float(davies_bouldin_score(X_pca, labels))
    else:
        row["silhouette"] = np.nan
        row["calinski_harabasz"] = np.nan
        row["davies_bouldin"] = np.nan
    score_rows.append(row)
score_df = pd.DataFrame(score_rows)

knee = KneeLocator(
    score_df["k"].to_numpy(),
    score_df["inertia"].to_numpy(),
    curve="convex",
    direction="decreasing",
).knee
if knee is None:
    elbow_points = score_df[["k", "inertia"]].to_numpy(float)
    first_point, last_point = elbow_points[0], elbow_points[-1]
    elbow_line = last_point - first_point
    distances = np.abs(np.cross(elbow_line, first_point - elbow_points)) / np.linalg.norm(elbow_line)
    distances[0] = -np.inf
    BEST_K = int(score_df.iloc[int(np.nanargmax(distances))]["k"])
    elbow_method = "maximum distance to endpoint line"
else:
    BEST_K = int(knee)
    elbow_method = "KneeLocator"

best_model = KMeans(
    n_clusters=BEST_K,
    random_state=RANDOM_STATE,
    n_init=KMEANS_N_INIT,
    init="k-means++",
).fit(X_pca)
kmeans_labels = best_model.labels_.astype(int)

# Keep Pattern I/II labels consistent with the existing manuscript result.
label_alignment = "native K-means labels"
if BEST_K == 2 and REFERENCE_RESULT_CSV.exists():
    reference_labels = pd.read_csv(REFERENCE_RESULT_CSV)[
        ["event", "cluster_id", "kmeans_cluster"]
    ].copy()
    alignment_table = center_df[["event", "cluster_id"]].copy()
    alignment_table["new_label"] = kmeans_labels
    alignment_table = alignment_table.merge(
        reference_labels,
        on=["event", "cluster_id"],
        how="left",
        validate="one_to_one",
    )
    direct_agreement = np.mean(
        alignment_table["new_label"].to_numpy()
        == alignment_table["kmeans_cluster"].to_numpy()
    )
    flipped_agreement = np.mean(
        (1 - alignment_table["new_label"].to_numpy())
        == alignment_table["kmeans_cluster"].to_numpy()
    )
    if flipped_agreement > direct_agreement:
        kmeans_labels = 1 - kmeans_labels
        label_alignment = f"binary labels flipped to reference ({flipped_agreement:.1%} agreement)"
    else:
        label_alignment = f"native labels matched reference ({direct_agreement:.1%} agreement)"

result_df = center_df[["event", "cluster_id"]].copy()
result_df["PC1"] = X_pca[:, 0]
result_df["PC2"] = X_pca[:, 1]
result_df["kmeans_cluster"] = kmeans_labels
result_df["pattern"] = result_df["kmeans_cluster"].map({0: "Pattern I", 1: "Pattern II"})
cluster_sizes = result_df["kmeans_cluster"].value_counts().sort_index().to_dict()

# SHAP explains a Random Forest surrogate of the K-means labels.
rf_surrogate = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    max_features="sqrt",
).fit(X_scaled, kmeans_labels)
shap_values = shap.TreeExplainer(rf_surrogate)(X_scaled).values
if shap_values.ndim == 3:
    global_mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2))
else:
    global_mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance_df = pd.DataFrame(
    {
        "feature": X_scaled.columns,
        "mean_abs_shap": global_mean_abs_shap,
    }
).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

pca_explained_df = pd.DataFrame(
    {
        "component": np.arange(1, len(cumulative_variance) + 1),
        "explained_variance_ratio": pca_full.explained_variance_ratio_,
        "cumulative_variance_ratio": cumulative_variance,
    }
)

print(f"Matched windows: {len(matched_df)}")
print(f"PCS-state centers: {len(center_df)}")
print(f"Features: {len(raw_feature_columns)} raw -> {X_scaled.shape[1]} retained")
print(
    f"PCA: {n_components} components retain "
    f"{pca.explained_variance_ratio_.sum():.3%} variance"
)
print(f"Elbow: K={BEST_K} ({elbow_method}); cluster sizes={cluster_sizes}")
print(f"Pattern-label alignment: {label_alignment}")
display(shap_importance_df.head(SHAP_TOP_N))

## Draw and export Figure S6

In [ ]:
# ======================== EDIT FIGURE S6 STYLE HERE ========================
FIGSIZE_MM = (183, 145)
add_font = 2
FONT = {
    "family": "Arial",
    "base": 8.0 +add_font ,
    "axis_label": 8.5+add_font,
    "axis_title": 9.0+add_font,
    "tick": 7.3+add_font,
    "legend": 7.1+add_font,
    "annotation": 7.0+add_font,
    "panel_label": 12.0+add_font,
}
LAYOUT = {
    "left": 0.085,
    "right": 0.975,
    "bottom": 0.080,
    "top": 0.955,
    "wspace": 0.52,
    "hspace": 0.34,
}
LINE_WIDTH = 1.6
MARKER_SIZE = 4.5
POINT_SIZE = 38
POINT_ALPHA = 0.86
GRID_ALPHA = 0.18
PANEL_LABEL_POSITION = (-0.16, 1.10)
PATTERN_COLORS = {0: "tab:blue", 1: "#E34040"}
SHAP_COLOR = "#7B4F9D"
OUTPUT_FORMATS = ("png", "tiff")
PNG_DPI = 400
TIFF_DPI = 600

# Labels used only for display; source tables retain the original feature names.
FEATURE_DISPLAY_NAMES = {
    "NBRPEAKFREQMAX": "Peak-freq. maximum",
    "DISTQ2Q1": "Q2-Q1 freq. distance",
    "NBRPEAKFREQCENTER": "Peak-freq. center",
    "NBRPEAKMEAN": "Mean peak count",
    "NBRPEAKMEDIAN": "Median peak count",
    "NBRPEAKMAX": "Maximum peak count",
    "gamma1": "Spectral shape, gamma1",
    "FmaxFFT": "Maximum FFT frequency",
    "rectilinearity": "Rectilinearity",
    "E1FFT": "FFT energy E1",
    "E4FFT": "FFT energy E4",
    "Energy_1_5Hz": "Energy at 1-5 Hz",
    "SpecKurtoMedianEnv": "Envelope spectral kurtosis",
    "RappMaxMean": "Maximum-to-mean ratio",
    "RATIONBRPEAKMAXMEAN": "Peak maximum-to-mean ratio",
    "INT_RATIO": "Integral ratio",
}
# ============================================================================

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": [FONT["family"], "Helvetica", "DejaVu Sans"],
        "font.size": FONT["base"],
        "axes.labelsize": FONT["axis_label"],
        "axes.titlesize": FONT["axis_title"],
        "xtick.labelsize": FONT["tick"],
        "ytick.labelsize": FONT["tick"],
        "legend.fontsize": FONT["legend"],
        "axes.linewidth": 0.75,
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(FIGSIZE_MM[0] / 25.4, FIGSIZE_MM[1] / 25.4),
    facecolor="white",
)
fig.subplots_adjust(**LAYOUT)
ax_pca, ax_elbow, ax_scatter, ax_shap = axes.ravel()

# --- a, PCA component selection ---
pca_x = np.arange(1, len(cumulative_variance) + 1)
ax_pca.plot(
    pca_x,
    cumulative_variance,
    color="#2F5D8A",
    linewidth=LINE_WIDTH,
    marker="o",
    markersize=MARKER_SIZE,
    markerfacecolor="white",
    markeredgewidth=0.8,
)
ax_pca.axhline(
    PCA_VARIANCE_THRESHOLD,
    color="#C44E52",
    linestyle="--",
    linewidth=1.0,
)
ax_pca.axvline(n_components, color="0.25", linestyle=":", linewidth=1.0)
ax_pca.scatter(
    [n_components],
    [cumulative_variance[n_components - 1]],
    s=45,
    color="#C44E52",
    edgecolor="white",
    linewidth=0.7,
    zorder=4,
)
ax_pca.annotate(
    f"{n_components} PCs; {cumulative_variance[n_components - 1]:.1%}",
    xy=(n_components, cumulative_variance[n_components - 1]),
    xytext=(n_components + 2.0, 0.82),
    fontsize=FONT["annotation"],
    arrowprops={"arrowstyle": "-", "color": "0.35", "lw": 0.7},
)
ax_pca.set_xlim(0.5, len(cumulative_variance) + 0.5)
ax_pca.set_ylim(0.25, 1.015)
ax_pca.set_xticks([1, 5, 10, 15, 20, 25, 30, 35])
ax_pca.set_xlabel("Number of principal components")
ax_pca.set_ylabel("Cumulative explained variance")
# --- b, elbow-based K selection with silhouette shown as a secondary diagnostic ---
ax_elbow.plot(
    score_df["k"],
    score_df["inertia"],
    color="#2F5D8A",
    linewidth=LINE_WIDTH,
    marker="o",
    markersize=MARKER_SIZE,
    label="WCSS",
)
ax_elbow.axvline(BEST_K, color="#C44E52", linestyle="--", linewidth=1.0)
selected_inertia = float(score_df.loc[score_df["k"].eq(BEST_K), "inertia"].iloc[0])
ax_elbow.scatter(
    [BEST_K],
    [selected_inertia],
    s=45,
    color="#C44E52",
    edgecolor="white",
    linewidth=0.7,
    zorder=4,
)
ax_elbow.annotate(
    f"Elbow: K={BEST_K}",
    xy=(BEST_K, selected_inertia),
    xytext=(BEST_K + 0.45, selected_inertia + 110),
    fontsize=FONT["annotation"],
    arrowprops={"arrowstyle": "-", "color": "0.35", "lw": 0.7},
)
ax_elbow.set_xticks(score_df["k"])
ax_elbow.set_xlabel("Number of clusters, K")
ax_elbow.set_ylabel("Within-cluster sum of squares")

handles_1, labels_1 = ax_elbow.get_legend_handles_labels()
ax_elbow.legend(handles_1 , labels_1 , loc="center right")

# --- c, PCA projection of the K-means patterns ---
for cluster_id in sorted(result_df["kmeans_cluster"].unique()):
    cluster_rows = result_df[result_df["kmeans_cluster"].eq(cluster_id)]
    ax_scatter.scatter(
        cluster_rows["PC1"],
        cluster_rows["PC2"],
        s=POINT_SIZE,
        color=PATTERN_COLORS[cluster_id],
        alpha=POINT_ALPHA,
        edgecolor="white",
        linewidth=0.55,
        label=f"Pattern {'I' if cluster_id == 0 else 'II'} (n={len(cluster_rows)})",
        zorder=3,
    )
    ax_scatter.scatter(
        [cluster_rows["PC1"].mean()],
        [cluster_rows["PC2"].mean()],
        s=95,
        marker="*",
        color=PATTERN_COLORS[cluster_id],
        edgecolor="black",
        linewidth=0.55,
        zorder=5,
    )

for row in result_df.itertuples(index=False):
    event_number = str(row.event).replace("event", "")
    ax_scatter.annotate(
        event_number,
        (row.PC1, row.PC2),
        xytext=(2.0, 1.8),
        textcoords="offset points",
        fontsize=5.2,
        color="0.25",
        alpha=0.85,
    )
ax_scatter.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax_scatter.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax_scatter.legend(loc="best", handletextpad=0.4)

# --- d, global SHAP importance of the Random Forest surrogate ---
top_shap = shap_importance_df.head(SHAP_TOP_N).copy().iloc[::-1]
top_shap["display_name"] = top_shap["feature"].map(
    lambda name: FEATURE_DISPLAY_NAMES.get(name, name)
)
ax_shap.barh(
    top_shap["display_name"],
    top_shap["mean_abs_shap"],
    color=SHAP_COLOR,
    alpha=0.88,
    edgecolor="white",
    linewidth=0.45,
)
ax_shap.set_xlabel("Mean |SHAP value|")
ax_shap.set_ylabel("")
ax_shap.tick_params(axis="y", labelsize=8)

# Shared panel styling and labels.
for panel_axis, panel_label in zip(
    [ax_pca, ax_elbow, ax_scatter, ax_shap],
    ["a", "b", "c", "d"],
):
    panel_axis.spines["top"].set_visible(False)
    panel_axis.spines["right"].set_visible(False)
    panel_axis.grid(axis="both", color="0.75", linewidth=0.45, alpha=GRID_ALPHA, zorder=0)
    panel_axis.text(
        PANEL_LABEL_POSITION[0],
        PANEL_LABEL_POSITION[1],
        panel_label,
        transform=panel_axis.transAxes,
        fontsize=FONT["panel_label"],
        fontweight="bold",
        ha="left",
        va="top",
    )
# ax_silhouette.spines["top"].set_visible(False)

# Export editable vector files and high-resolution rasters.
figure_stem = OUT_DIR / "FigureS6_second_clustering"
figure_paths = []
for extension in OUTPUT_FORMATS:
    output_path = figure_stem.with_suffix(f".{extension}")
    save_options = {
        "bbox_inches": "tight",
        "facecolor": "white",
        "pad_inches": 0.04,
    }
    if extension == "png":
        save_options["dpi"] = PNG_DPI
    elif extension == "tiff":
        save_options["dpi"] = TIFF_DPI
        save_options["pil_kwargs"] = {"compression": "tiff_lzw"}
    fig.savefig(output_path, **save_options)
    figure_paths.append(output_path)
    print(output_path)

source_data_path = OUT_DIR / "FigureS6_second_clustering_source_data.xlsx"
with pd.ExcelWriter(source_data_path, engine="openpyxl") as writer:
    pca_explained_df.to_excel(writer, sheet_name="pca_variance", index=False)
    score_df.to_excel(writer, sheet_name="cluster_number_scores", index=False)
    result_df.to_excel(writer, sheet_name="pca_scatter", index=False)
    shap_importance_df.to_excel(writer, sheet_name="shap_importance", index=False)
    pd.DataFrame(
        {
            "parameter": [
                "PCA variance threshold", "retained components",
                "retained cumulative variance", "selected K", "elbow method",
                "retained feature count", "Pattern I states", "Pattern II states",
            ],
            "value": [
                PCA_VARIANCE_THRESHOLD, n_components,
                pca.explained_variance_ratio_.sum(), BEST_K, elbow_method,
                X_scaled.shape[1], cluster_sizes.get(0, 0), cluster_sizes.get(1, 0),
            ],
        }
    ).to_excel(writer, sheet_name="summary", index=False)

display(fig)
plt.close(fig)
print("Figure S6 exports:", [path.name for path in figure_paths])
print("Source data:", source_data_path.name)


## Interpretation boundary

- The 95% cumulative-variance rule determines the number of PCA components used
  by K-means; the scatter plot shows only the first two components and is therefore
  a projection rather than the full clustering space.
- The elbow criterion selects the number of clusters. Silhouette is shown as a
  secondary diagnostic and was not used to override the elbow selection.
- SHAP values explain the Random Forest surrogate's reproduction of K-means labels.
  They rank discriminatory features but do not identify causal controls on debris-flow
  initiation.

# 最终版

In [ ]:
# ======================== EDIT FIGURE S6 STYLE HERE ========================
FIGSIZE_MM = (183, 145)
add_font = 2
FONT = {
    "family": "Arial",
    "base": 8.0 +add_font ,
    "axis_label": 8.5+add_font,
    "axis_title": 9.0+add_font,
    "tick": 7.3+add_font,
    "legend": 7.1+add_font,
    "annotation": 7.0+add_font,
    "panel_label": 12.0+add_font,
}
LAYOUT = {
    "left": 0.085,
    "right": 0.975,
    "bottom": 0.080,
    "top": 0.955,
    "wspace": 0.52,
    "hspace": 0.34,
}
LINE_WIDTH = 1.6
MARKER_SIZE = 4.5
POINT_SIZE = 38
POINT_ALPHA = 0.86
GRID_ALPHA = 0.18
PANEL_LABEL_POSITION = (-0.16, 1.10)
PATTERN_COLORS = {0: "tab:blue", 1: "#E34040"}
SHAP_COLOR = "#7B4F9D"
OUTPUT_FORMATS = ("png", "tiff")
PNG_DPI = 400
TIFF_DPI = 600

# Labels used only for display; source tables retain the original feature names.
FEATURE_DISPLAY_NAMES = {
    "NBRPEAKFREQMAX": "Peak-freq. maximum",
    "DISTQ2Q1": "Q2-Q1 freq. distance",
    "NBRPEAKFREQCENTER": "Peak-freq. center",
    "NBRPEAKMEAN": "Mean peak count",
    "NBRPEAKMEDIAN": "Median peak count",
    "NBRPEAKMAX": "Maximum peak count",
    "gamma1": "Spectral shape, gamma1",
    "FmaxFFT": "Maximum FFT frequency",
    "rectilinearity": "Rectilinearity",
    "E1FFT": "FFT energy E1",
    "E4FFT": "FFT energy E4",
    "Energy_1_5Hz": "Energy at 1-5 Hz",
    "SpecKurtoMedianEnv": "Envelope spectral kurtosis",
    "RappMaxMean": "Maximum-to-mean ratio",
    "RATIONBRPEAKMAXMEAN": "Peak maximum-to-mean ratio",
    "INT_RATIO": "Integral ratio",
}
# ============================================================================

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": [FONT["family"], "Helvetica", "DejaVu Sans"],
        "font.size": FONT["base"],
        "axes.labelsize": FONT["axis_label"],
        "axes.titlesize": FONT["axis_title"],
        "xtick.labelsize": FONT["tick"],
        "ytick.labelsize": FONT["tick"],
        "legend.fontsize": FONT["legend"],
        "axes.linewidth": 0.75,
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(FIGSIZE_MM[0] / 25.4, FIGSIZE_MM[1] / 25.4),
    facecolor="white",
)
fig.subplots_adjust(**LAYOUT)
ax_pca, ax_elbow, ax_scatter, ax_shap = axes.ravel()

# --- a, PCA component selection ---
pca_x = np.arange(1, len(cumulative_variance) + 1)
ax_pca.plot(
    pca_x,
    cumulative_variance,
    color="#2F5D8A",
    linewidth=LINE_WIDTH,
    marker="o",
    markersize=MARKER_SIZE,
    markerfacecolor="white",
    markeredgewidth=0.8,
)
ax_pca.axhline(
    PCA_VARIANCE_THRESHOLD,
    color="#C44E52",
    linestyle="--",
    linewidth=1.0,
)
ax_pca.axvline(n_components, color="0.25", linestyle=":", linewidth=1.0)
ax_pca.scatter(
    [n_components],
    [cumulative_variance[n_components - 1]],
    s=45,
    color="#C44E52",
    edgecolor="white",
    linewidth=0.7,
    zorder=4,
)
ax_pca.annotate(
    f"{n_components} PCs; {cumulative_variance[n_components - 1]:.1%}",
    xy=(n_components, cumulative_variance[n_components - 1]),
    xytext=(n_components + 2.0, 0.82),
    fontsize=FONT["annotation"],
    arrowprops={"arrowstyle": "-", "color": "0.35", "lw": 0.7},
)
ax_pca.set_xlim(0.5, len(cumulative_variance) + 0.5)
ax_pca.set_ylim(0.25, 1.015)
ax_pca.set_xticks([1, 5, 10, 15, 20, 25, 30, 35])
ax_pca.set_xlabel("Number of principal components")
ax_pca.set_ylabel("Cumulative explained variance")
# --- b, elbow-based K selection with silhouette shown as a secondary diagnostic ---
ax_elbow.plot(
    score_df["k"],
    score_df["inertia"],
    color="#2F5D8A",
    linewidth=LINE_WIDTH,
    marker="o",
    markersize=MARKER_SIZE,
    label="WCSS",
)
ax_elbow.axvline(BEST_K, color="#C44E52", linestyle="--", linewidth=1.0)
selected_inertia = float(score_df.loc[score_df["k"].eq(BEST_K), "inertia"].iloc[0])
ax_elbow.scatter(
    [BEST_K],
    [selected_inertia],
    s=45,
    color="#C44E52",
    edgecolor="white",
    linewidth=0.7,
    zorder=4,
)
ax_elbow.annotate(
    f"Elbow: K={BEST_K}",
    xy=(BEST_K, selected_inertia),
    xytext=(BEST_K + 0.45, selected_inertia + 110),
    fontsize=FONT["annotation"],
    arrowprops={"arrowstyle": "-", "color": "0.35", "lw": 0.7},
)
ax_elbow.set_xticks(score_df["k"])
ax_elbow.set_xlabel("Number of clusters, K")
ax_elbow.set_ylabel("Within-cluster sum of squares")

handles_1, labels_1 = ax_elbow.get_legend_handles_labels()
ax_elbow.legend(handles_1 , labels_1 , loc="center right")

# --- c, PCA projection of the K-means patterns ---
for cluster_id in sorted(result_df["kmeans_cluster"].unique()):
    cluster_rows = result_df[result_df["kmeans_cluster"].eq(cluster_id)]
    ax_scatter.scatter(
        cluster_rows["PC1"],
        cluster_rows["PC2"],
        s=POINT_SIZE,
        color=PATTERN_COLORS[cluster_id],
        alpha=POINT_ALPHA,
        edgecolor="white",
        linewidth=0.55,
        label=f"Pattern {'I' if cluster_id == 0 else 'II'} (n={len(cluster_rows)})",
        zorder=3,
    )
    ax_scatter.scatter(
        [cluster_rows["PC1"].mean()],
        [cluster_rows["PC2"].mean()],
        s=95,
        marker="*",
        color=PATTERN_COLORS[cluster_id],
        edgecolor="black",
        linewidth=0.55,
        zorder=5,
    )

for row in result_df.itertuples(index=False):
    event_number = str(row.event).replace("event", "")
    ax_scatter.annotate(
        event_number,
        (row.PC1, row.PC2),
        xytext=(2.0, 1.8),
        textcoords="offset points",
        fontsize=5.2,
        color="0.25",
        alpha=0.85,
    )
ax_scatter.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax_scatter.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax_scatter.legend(loc="best", handletextpad=0.4)

# --- d, global SHAP importance of the Random Forest surrogate ---
top_shap = shap_importance_df.head(SHAP_TOP_N).copy().iloc[::-1]
top_shap["display_name"] = top_shap["feature"].map(
    lambda name: FEATURE_DISPLAY_NAMES.get(name, name)
)
ax_shap.barh(
    top_shap["display_name"],
    top_shap["mean_abs_shap"],
    color=SHAP_COLOR,
    alpha=0.88,
    edgecolor="white",
    linewidth=0.45,
)
ax_shap.set_xlabel("Mean |SHAP value|")
ax_shap.set_ylabel("")
ax_shap.tick_params(axis="y", labelsize=8)

# Shared panel styling and labels.
for panel_axis, panel_label in zip(
    [ax_pca, ax_elbow, ax_scatter, ax_shap],
    ["a", "b", "c", "d"],
):
    panel_axis.spines["top"].set_visible(False)
    panel_axis.spines["right"].set_visible(False)
    panel_axis.grid(axis="both", color="0.75", linewidth=0.45, alpha=GRID_ALPHA, zorder=0)
    panel_axis.text(
        PANEL_LABEL_POSITION[0],
        PANEL_LABEL_POSITION[1],
        panel_label,
        transform=panel_axis.transAxes,
        fontsize=FONT["panel_label"],
        fontweight="bold",
        ha="left",
        va="top",
    )
# ax_silhouette.spines["top"].set_visible(False)

# Export editable vector files and high-resolution rasters.
for ax in (ax_pca,ax_elbow,ax_scatter,ax_shap):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.75)
    ax.tick_params(axis="both",which="both",direction="out",bottom=True,left=True,top=False,right=False,labeltop=False,labelright=False)
    ax.tick_params(axis="both",which="major",length=3,width=0.65)
    
    
    
from pathlib import Path
save_root=Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all")
(save_root/"png").mkdir(parents=True,exist_ok=True)
(save_root/"tiff").mkdir(parents=True,exist_ok=True)
png_path=save_root/"png"/"Fig S7.png"
tiff_path=save_root/"tiff"/"Fig S7.tiff"
fig.savefig(png_path,format="png",dpi=PNG_DPI,bbox_inches="tight",facecolor="white",pad_inches=0.04)
fig.savefig(tiff_path,format="tiff",dpi=TIFF_DPI,bbox_inches="tight",facecolor="white",pad_inches=0.04,pil_kwargs={"compression":"tiff_lzw"})
figure_paths=[png_path,tiff_path]

with pd.ExcelWriter(source_data_path, engine="openpyxl") as writer:
    pca_explained_df.to_excel(writer, sheet_name="pca_variance", index=False)
    score_df.to_excel(writer, sheet_name="cluster_number_scores", index=False)
    result_df.to_excel(writer, sheet_name="pca_scatter", index=False)
    shap_importance_df.to_excel(writer, sheet_name="shap_importance", index=False)
    pd.DataFrame(
        {
            "parameter": [
                "PCA variance threshold", "retained components",
                "retained cumulative variance", "selected K", "elbow method",
                "retained feature count", "Pattern I states", "Pattern II states",
            ],
            "value": [
                PCA_VARIANCE_THRESHOLD, n_components,
                pca.explained_variance_ratio_.sum(), BEST_K, elbow_method,
                X_scaled.shape[1], cluster_sizes.get(0, 0), cluster_sizes.get(1, 0),
            ],
        }
    ).to_excel(writer, sheet_name="summary", index=False)

display(fig)
plt.close(fig)
print("Figure S7 exports:", [path.name for path in figure_paths])
print("Source data:", source_data_path.name)


In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# ---------- 颜色：统一在这里调整 ----------
COLORS={
    "pattern_i":"tab:blue","pattern_ii":"#E34040","pca_curve":"#2F5D8A","elbow_curve":"#2F5D8A",
    "threshold":"#C44E52","selected_point":"#C44E52","component_line":"0.25","annotation_arrow":"0.35",
    "event_text":"0.25","shap_bar":"#8dace1","grid":"0.75","frame":"black","tick":"black",
    "text":"black","panel_label":"black","centroid_edge":"black","point_edge":"white","background":"white",
}
PATTERN_COLORS={0:COLORS["pattern_i"],1:COLORS["pattern_ii"]}

# ---------- 字体、尺寸及间距 ----------
FIGSIZE_MM=(183,145)
add_font=2
FONT={"family":"Arial","base":8.0+add_font,"axis_label":8.5+add_font,"axis_title":9.0+add_font,"tick":7.3+add_font,"legend":7.1+add_font,"annotation":7.0+add_font,"panel_label":12.0+add_font,"event_label":5.2,"shap_tick":8}
LAYOUT={"left":0.085,"right":0.975,"bottom":0.080,"top":0.955,"wspace":0.52,"hspace":0.34}
LINE_WIDTH=1.6
MARKER_SIZE=4.5
POINT_SIZE=38
POINT_ALPHA=0.86
GRID_ALPHA=0.18
PANEL_LABEL_POSITION=(-0.16,1.10)
PNG_DPI=400
TIFF_DPI=600
SAVE_PAD=0.04
SAVE_ROOT=Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all")
FEATURE_DISPLAY_NAMES={
    "NBRPEAKFREQMAX":"Peak-freq. maximum","DISTQ2Q1":"Q2-Q1 freq. distance","NBRPEAKFREQCENTER":"Peak-freq. center",
    "NBRPEAKMEAN":"Mean peak count","NBRPEAKMEDIAN":"Median peak count","NBRPEAKMAX":"Maximum peak count",
    "gamma1":"Spectral shape, gamma1","FmaxFFT":"Maximum FFT frequency","rectilinearity":"Rectilinearity",
    "E1FFT":"FFT energy E1","E4FFT":"FFT energy E4","Energy_1_5Hz":"Energy at 1-5 Hz",
    "SpecKurtoMedianEnv":"Envelope spectral kurtosis","RappMaxMean":"Maximum-to-mean ratio",
    "RATIONBRPEAKMAXMEAN":"Peak maximum-to-mean ratio","INT_RATIO":"Integral ratio",
}
STYLE={"font.family":"sans-serif","font.sans-serif":[FONT["family"],"Helvetica","DejaVu Sans"],"font.size":FONT["base"],"axes.labelsize":FONT["axis_label"],"axes.titlesize":FONT["axis_title"],"xtick.labelsize":FONT["tick"],"ytick.labelsize":FONT["tick"],"legend.fontsize":FONT["legend"],"axes.linewidth":0.75,"axes.spines.top":True,"axes.spines.right":True,"axes.spines.bottom":True,"axes.spines.left":True,"axes.edgecolor":COLORS["frame"],"axes.labelcolor":COLORS["text"],"text.color":COLORS["text"],"xtick.color":COLORS["tick"],"ytick.color":COLORS["tick"],"axes.facecolor":COLORS["background"],"figure.facecolor":COLORS["background"],"xtick.direction":"out","ytick.direction":"out","xtick.top":False,"ytick.right":False,"legend.frameon":False,"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none"}

# ---------- 保存目录 ----------
(SAVE_ROOT/"png").mkdir(parents=True,exist_ok=True)
(SAVE_ROOT/"tiff").mkdir(parents=True,exist_ok=True)
png_path=SAVE_ROOT/"png"/"Fig S7.png"
tiff_path=SAVE_ROOT/"tiff"/"Fig S7.tiff"
OUT_DIR=Path(OUT_DIR)
OUT_DIR.mkdir(parents=True,exist_ok=True)
source_data_path=OUT_DIR/"FigureS7_second_clustering_source_data.xlsx"

# ---------- 绘图：使用已有的数据与模型变量 ----------
with mpl.rc_context(STYLE):
    fig,axes=plt.subplots(2,2,figsize=(FIGSIZE_MM[0]/25.4,FIGSIZE_MM[1]/25.4),facecolor=COLORS["background"])
    fig.subplots_adjust(**LAYOUT)
    ax_pca,ax_elbow,ax_scatter,ax_shap=axes.ravel()

    # a：PCA主成分选择
    pca_x=np.arange(1,len(cumulative_variance)+1)
    ax_pca.plot(pca_x,cumulative_variance,color=COLORS["pca_curve"],linewidth=LINE_WIDTH,marker="o",markersize=MARKER_SIZE,markerfacecolor=COLORS["point_edge"],markeredgewidth=0.8)
    ax_pca.axhline(PCA_VARIANCE_THRESHOLD,color=COLORS["threshold"],linestyle="--",linewidth=1.0)
    ax_pca.axvline(n_components,color=COLORS["component_line"],linestyle=":",linewidth=1.0)
    ax_pca.scatter([n_components],[cumulative_variance[n_components-1]],s=45,color=COLORS["selected_point"],edgecolor=COLORS["point_edge"],linewidth=0.7,zorder=4)
    ax_pca.annotate(f"{n_components} PCs; {cumulative_variance[n_components-1]:.1%}",xy=(n_components,cumulative_variance[n_components-1]),xytext=(n_components+2.0,0.82),fontsize=FONT["annotation"],arrowprops={"arrowstyle":"-","color":COLORS["annotation_arrow"],"lw":0.7})
    ax_pca.set_xticks([1,5,10,15,20,25,30,35])
    ax_pca.set_xlim(0.5,len(cumulative_variance)+0.5)
    ax_pca.set_ylim(0.25,1.015)
    ax_pca.set_xlabel("Number of principal components")
    ax_pca.set_ylabel("Cumulative explained variance")

    # b：肘部法选择K
    ax_elbow.plot(score_df["k"],score_df["inertia"],color=COLORS["elbow_curve"],linewidth=LINE_WIDTH,marker="o",markersize=MARKER_SIZE,label="WCSS")
    ax_elbow.axvline(BEST_K,color=COLORS["threshold"],linestyle="--",linewidth=1.0)
    selected_inertia=float(score_df.loc[score_df["k"].eq(BEST_K),"inertia"].iloc[0])
    ax_elbow.scatter([BEST_K],[selected_inertia],s=45,color=COLORS["selected_point"],edgecolor=COLORS["point_edge"],linewidth=0.7,zorder=4)
    ax_elbow.annotate(f"Elbow: K={BEST_K}",xy=(BEST_K,selected_inertia),xytext=(BEST_K+0.45,selected_inertia+110),fontsize=FONT["annotation"],arrowprops={"arrowstyle":"-","color":COLORS["annotation_arrow"],"lw":0.7})
    ax_elbow.set_xticks(score_df["k"])
    ax_elbow.set_xlabel("Number of clusters, K")
    ax_elbow.set_ylabel("Within-cluster sum of squares")
    ax_elbow.legend(loc="center right")

    # c：两类模式的PCA投影
    for cluster_id in sorted(result_df["kmeans_cluster"].unique()):
        cluster_rows=result_df[result_df["kmeans_cluster"].eq(cluster_id)]
        ax_scatter.scatter(cluster_rows["PC1"],cluster_rows["PC2"],s=POINT_SIZE,color=PATTERN_COLORS[cluster_id],alpha=POINT_ALPHA,edgecolor=COLORS["point_edge"],linewidth=0.55,label=f"Pattern {'I' if cluster_id==0 else 'II'} (n={len(cluster_rows)})",zorder=3)
        ax_scatter.scatter([cluster_rows["PC1"].mean()],[cluster_rows["PC2"].mean()],s=95,marker="*",color=PATTERN_COLORS[cluster_id],edgecolor=COLORS["centroid_edge"],linewidth=0.55,zorder=5)
    for row in result_df.itertuples(index=False):
        event_number=str(row.event).replace("event","")
        ax_scatter.annotate(event_number,(row.PC1,row.PC2),xytext=(2.0,1.8),textcoords="offset points",fontsize=FONT["event_label"],color=COLORS["event_text"],alpha=0.85)
    ax_scatter.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    ax_scatter.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    ax_scatter.legend(loc="best",handletextpad=0.4)

    # d：随机森林代理模型的SHAP重要性
    top_shap=shap_importance_df.head(SHAP_TOP_N).copy().iloc[::-1]
    top_shap["display_name"]=top_shap["feature"].map(lambda name:FEATURE_DISPLAY_NAMES.get(name,name))
    ax_shap.barh(top_shap["display_name"],top_shap["mean_abs_shap"],color=COLORS["shap_bar"],alpha=0.88,edgecolor=COLORS["point_edge"],linewidth=0.45)
    ax_shap.set_xlabel("Mean |SHAP value|")
    ax_shap.set_ylabel("")
    ax_shap.tick_params(axis="y",labelsize=FONT["shap_tick"])

    # 全框线、向外刻度及面板标签
    for ax,panel_label in zip((ax_pca,ax_elbow,ax_scatter,ax_shap),("a","b","c","d")):
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color(COLORS["frame"])
            spine.set_linewidth(0.75)
        ax.tick_params(axis="both",which="both",direction="out",bottom=True,left=True,top=False,right=False,labeltop=False,labelright=False)
        ax.tick_params(axis="both",which="major",length=3,width=0.65)
        ax.grid(axis="both",color=COLORS["grid"],linewidth=0.45,alpha=GRID_ALPHA,zorder=0)
        ax.text(*PANEL_LABEL_POSITION,panel_label,transform=ax.transAxes,fontsize=FONT["panel_label"],fontweight="bold",ha="left",va="top",color=COLORS["panel_label"])

    # 保存图片
    fig.savefig(png_path,format="png",dpi=PNG_DPI,bbox_inches="tight",facecolor=COLORS["background"],pad_inches=SAVE_PAD)
    fig.savefig(tiff_path,format="tiff",dpi=TIFF_DPI,bbox_inches="tight",facecolor=COLORS["background"],pad_inches=SAVE_PAD,pil_kwargs={"compression":"tiff_lzw"})
    display(fig)
    plt.close(fig)

# ---------- 保存源数据 ----------
with pd.ExcelWriter(source_data_path,engine="openpyxl") as writer:
    pca_explained_df.to_excel(writer,sheet_name="pca_variance",index=False)
    score_df.to_excel(writer,sheet_name="cluster_number_scores",index=False)
    result_df.to_excel(writer,sheet_name="pca_scatter",index=False)
    shap_importance_df.to_excel(writer,sheet_name="shap_importance",index=False)
    pd.DataFrame({"parameter":["PCA variance threshold","retained components","retained cumulative variance","selected K","elbow method","retained feature count","Pattern I states","Pattern II states"],"value":[PCA_VARIANCE_THRESHOLD,n_components,pca.explained_variance_ratio_.sum(),BEST_K,elbow_method,X_scaled.shape[1],cluster_sizes.get(0,0),cluster_sizes.get(1,0)]}).to_excel(writer,sheet_name="summary",index=False)

print("Figure S7 exports:")
print(png_path)
print(tiff_path)
print("Source data:",source_data_path)